In [1]:
!pip install -q transformers accelerate sentencepiece pandas tqdm


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
!git merge --abort

In [12]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .ipynb_checkpoints/Output_Generation-checkpoint.ipynb
	deleted:    .ipynb_checkpoints/training_pairs-checkpoint.jsonl
	deleted:    .ipynb_checkpoints/training_pairs_50-checkpoint.jsonl
	deleted:    .ipynb_checkpoints/training_pairs_rocm-checkpoint.jsonl
	modified:   Data_Preprocessing.ipynb
	deleted:    Output_Generation.ipynb
	deleted:    fix_generation_prompts.jsonl
	deleted:    training_pairs.jsonl
	deleted:    training_pairs_50.jsonl
	deleted:    training_pairs_rocm.jsonl

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.ipynb_checkpoints/Dataset_Generation-checkpoint.ipynb
	.ipynb_checkpoints/Model_Finetuning-checkpoint.ipynb
	Dataset_Generation.ipynb
	Model_Finetuning.ipynb
	data/eval_detect_fix.jsonl
	data/

In [13]:
!git add data

In [10]:
!git commit -m "model creation complete"

On branch main
Your branch and 'origin/main' have diverged,
and have 7 and 3 different commits each, respectively.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .ipynb_checkpoints/Output_Generation-checkpoint.ipynb
	deleted:    .ipynb_checkpoints/training_pairs-checkpoint.jsonl
	deleted:    .ipynb_checkpoints/training_pairs_50-checkpoint.jsonl
	deleted:    .ipynb_checkpoints/training_pairs_rocm-checkpoint.jsonl
	modified:   Dataset_Generation.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.ipynb_checkpoints/Dataset_Generation-checkpoint.ipynb
	qwen_java_security/
	qwen_security_lora/
	qwen_security_merged/
	vulnerable_only.csv

no changes added to commit (use "git add" and/or "git commit -a")


In [7]:
!git config --global user.name "ChaitanyaParab11"
!git config --global user.email "chaitanyaparab111@gmail.com"

In [16]:
!git push origin main

Username for 'https://github.com': ^C


In [2]:
import json
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True
)

print("Model loaded")

# ---------------------------------------
# LOAD DATA
# ---------------------------------------

df = pd.read_csv("data/eval_vulnerable.csv")

# PILOT RUN
# df = df.head(50)

print("Samples:", len(df))

# ---------------------------------------
# GENERATE FIXES
# ---------------------------------------

output_file = "eval_pairs.jsonl"

with open(output_file, "w", encoding="utf-8") as outfile:

    for idx, row in tqdm(df.iterrows(), total=len(df)):

        category = str(row["category"])
        cwe = str(row["cwe"])
        code = str(row["source_code"])

        prompt = f"""
You are a senior Java security engineer.

Vulnerability Category: {category}
CWE: CWE-{cwe}

Fix the security vulnerability.

Requirements:
1. Preserve functionality.
2. Use secure coding practices.
3. Return ONLY complete corrected Java code.
4. No markdown.
5. No explanations.

Java Code:

{code}
"""

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=12000
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=2500,
            do_sample=False,
            temperature=0.0
        )

        generated_text = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        generated_text = generated_text.replace(
            "```java",
            ""
        )
        
        generated_text = generated_text.replace(
            "```",
            ""
        )
        
        generated_text = generated_text.strip()

        record = {
            "instruction": f"Fix CWE-{cwe} {category} vulnerability",
            "input": code,
            "output": generated_text
        }

        outfile.write(
            json.dumps(record, ensure_ascii=False)
            + "\n"
        )

        if (idx + 1) % 5 == 0:
            print(f"Completed {idx + 1}")

print("Done")
print("Saved:", output_file)

Loading tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Completed 205



 99%|█████████▊| 210/213 [56:46<00:53, 17.94s/it]

Completed 210



100%|██████████| 213/213 [57:47<00:00, 16.28s/it]

Done
Saved: eval_pairs.jsonl


In [4]:
import json
import re

def clean_code(code):

    # remove markdown
    code = code.replace("```java", "")
    code = code.replace("```", "")

    # remove block comments
    code = re.sub(
        r"/\*.*?\*/",
        "",
        code,
        flags=re.DOTALL
    )

    # remove package line
    code = re.sub(
        r"package\s+[^\n;]+;",
        "",
        code
    )

    return code.strip()

for infile, outfile in [
    ("training_pairs.jsonl", "training_pairs_clean.jsonl"),
    ("eval_pairs.jsonl", "eval_pairs_clean.jsonl")
]:

    with open(infile, "r", encoding="utf-8") as fin, \
         open(outfile, "w", encoding="utf-8") as fout:

        for line in fin:

            row = json.loads(line)

            row["input"] = clean_code(row["input"])
            row["output"] = clean_code(row["output"])

            fout.write(
                json.dumps(row, ensure_ascii=False)
                + "\n"
            )

print("Done")

Done


In [5]:
import json

def convert(src, dst):

    with open(src, "r", encoding="utf-8") as fin, \
         open(dst, "w", encoding="utf-8") as fout:

        for line in fin:

            row = json.loads(line)

            text = f"""<|im_start|>user
{row['instruction']}

{row['input']}
<|im_end|>
<|im_start|>assistant
{row['output']}
<|im_end|>
"""

            fout.write(
                json.dumps(
                    {"text": text},
                    ensure_ascii=False
                )
                + "\n"
            )

convert(
    "training_pairs_clean.jsonl",
    "train_qwen.jsonl"
)

convert(
    "eval_pairs_clean.jsonl",
    "eval_qwen.jsonl"
)

print("Conversion complete")

Conversion complete


In [1]:
import json
 
CATEGORY_MAP = {
    "sqli": "SQL Injection",
    "xss": "Cross Site Scripting",
    "cmdi": "Command Injection",
    "pathtraver": "Path Traversal",
    "weakrand": "Weak Randomness",
    "crypto": "Weak Cryptography",
    "hash": "Weak Hashing",
    "trustbound": "Trust Boundary Violation",
    "securecookie": "Insecure Cookie Handling",
    "ldapi": "LDAP Injection",
    "xpathi": "XPath Injection"
}
 
with open("training_pairs.jsonl") as fin, \
     open("train_detect_fix.jsonl","w") as fout:
 
    for line in fin:
 
        item = json.loads(line)
 
        vuln = item["instruction"]
 
        category = None
 
        for key in CATEGORY_MAP:
            if key in vuln.lower():
                category = CATEGORY_MAP[key]
                break
 
        if category is None:
            continue
 
        text = f"""### Task
Analyze the Java code and fix the vulnerability.
 
### Java Code
 
{item["input"]}
 
### Vulnerability
 
{category}
 
### Secure Code
 
{item["output"]}
"""
 
        fout.write(
            json.dumps({"text": text}) + "\n"
        )
 
print("Done")
 

Done


In [2]:
import json
 
CATEGORY_MAP = {
    "sqli": "SQL Injection",
    "xss": "Cross Site Scripting",
    "cmdi": "Command Injection",
    "pathtraver": "Path Traversal",
    "weakrand": "Weak Randomness",
    "crypto": "Weak Cryptography",
    "hash": "Weak Hashing",
    "trustbound": "Trust Boundary Violation",
    "securecookie": "Insecure Cookie Handling",
    "ldapi": "LDAP Injection",
    "xpathi": "XPath Injection"
}
 
with open("eval_pairs.jsonl") as fin, \
     open("eval_detect_fix.jsonl","w") as fout:
 
    for line in fin:
 
        item = json.loads(line)
 
        vuln = item["instruction"]
 
        category = None
 
        for key in CATEGORY_MAP:
            if key in vuln.lower():
                category = CATEGORY_MAP[key]
                break
 
        if category is None:
            continue
 
        text = f"""### Task
Analyze the Java code and fix the vulnerability.
 
### Java Code
 
{item["input"]}
 
### Vulnerability
 
{category}
 
### Secure Code
 
{item["output"]}
"""
 
        fout.write(
            json.dumps({"text": text}) + "\n"
        )
 
print("Done")
 

Done


In [10]:
import json
import re
 
INPUT_FILE = "train_final.jsonl"
OUTPUT_FILE = "train_final_clean.jsonl"
 
def clean_text(text):
 
    patterns = [
 
        r"/\*\*.*?\*/",
 
        r"package\s+org\.owasp\.benchmark\.testcode\s*;",
 
        r"@WebServlet\s*\([^)]*\)",
 
        r"private\s+static\s+final\s+long\s+serialVersionUID\s*=.*?;",
 
        r"BenchmarkTest\d{5}",
 
        r"org\.owasp\.benchmark\.helpers\.",
 
        r"org\.owasp\.benchmark\.",
 
        r"org\.owasp\.esapi\."
    ]
 
    for p in patterns:
        text = re.sub(
            p,
            "",
            text,
            flags=re.DOTALL
        )
 
    text = re.sub(
        r"\n\s*\n\s*\n+",
        "\n\n",
        text
    )
 
    return text.strip()
 
count = 0
 
with open(INPUT_FILE) as fin, \
     open(OUTPUT_FILE, "w") as fout:
 
    for line in fin:
 
        sample = json.loads(line)
 
        sample["text"] = clean_text(
            sample["text"]
        )
 
        fout.write(
            json.dumps(sample) + "\n"
        )
 
        count += 1
 
print("Processed:", count)
 

Processed: 1502
